# BERTopic Modules

This notebook applies the `MODULE PIPELINE`-style embedding step from `bertopic_combined.ipynb` to the three cleaned university module CSV files.

## Configuration & File Paths

In [ ]:
import os
from pathlib import Path

# Get notebook directory and construct project path
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
PROJECT_DIR = os.path.join(NOTEBOOK_DIR, "..", "DSA4264 Project")

# Input module CSV files
NUS_MODULES_CSV = os.path.join(PROJECT_DIR, "all_nus_modules.csv")
SMU_MODULES_CSV = os.path.join(PROJECT_DIR, "all_smu_modules.csv")
SUTD_MODULES_CSV = os.path.join(PROJECT_DIR, "all_sutd_modules.csv")

# Output parquet files
NUS_OUTPUT_FILE = os.path.join(PROJECT_DIR, "all_nus_modules_embedded.parquet")
SMU_OUTPUT_FILE = os.path.join(PROJECT_DIR, "all_smu_modules_embedded.parquet")
SUTD_OUTPUT_FILE = os.path.join(PROJECT_DIR, "all_sutd_modules_embedded.parquet")

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Project directory: {PROJECT_DIR}")
print(f"NUS modules CSV: {NUS_MODULES_CSV}")
print(f"SMU modules CSV: {SMU_MODULES_CSV}")
print(f"SUTD modules CSV: {SUTD_MODULES_CSV}")

# MODULE PIPELINE

Flow adapted from `bertopic_combined.ipynb`:
- Load cleaned university module CSVs
- Standardize to `code`, `title`, and `description`
- Embed descriptions with MPNet
- Save one embedded parquet per university

In [ ]:
from pathlib import Path

import pandas as pd
from sentence_transformers import SentenceTransformer


def clean_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def resolve_mpnet_model_path():
    snapshots_dir = (
        Path.home()
        / ".cache"
        / "huggingface"
        / "hub"
        / "models--sentence-transformers--all-mpnet-base-v2"
        / "snapshots"
    )
    if snapshots_dir.exists():
        snapshots = sorted(path for path in snapshots_dir.iterdir() if path.is_dir())
        if snapshots:
            return str(snapshots[-1])
    return "all-mpnet-base-v2"


def standardize_modules(df, university):
    df = df.copy()
    df.columns = [str(col).strip().lower() for col in df.columns]

    if "module_code" not in df.columns:
        raise KeyError(f"{university}: expected 'module_code' column, found {df.columns.tolist()}")
    if "module_name" not in df.columns:
        raise KeyError(f"{university}: expected 'module_name' column, found {df.columns.tolist()}")
    if "description" not in df.columns:
        raise KeyError(f"{university}: expected 'description' column, found {df.columns.tolist()}")

    df["code"] = df["module_code"].map(clean_text)
    df["title"] = df["module_name"].map(clean_text)
    df["description"] = df["description"].map(clean_text)
    df["source_university"] = university.upper()

    df = df[df["code"].notna()].copy()
    df = df[df["description"].notna()].copy()
    df = df[df["description"] != ""].copy()
    df = df.drop_duplicates(subset=["code"]).reset_index(drop=True)

    front_cols = ["source_university", "code", "title", "description"]
    remaining_cols = [col for col in df.columns if col not in front_cols]
    return df[front_cols + remaining_cols]


def build_embedded_modules(csv_path, output_file, university, embedding_model):
    df = pd.read_csv(csv_path)
    standardized_df = standardize_modules(df, university)

    embeddings = embedding_model.encode(
        standardized_df["description"].tolist(),
        batch_size=32,
        show_progress_bar=True,
    )

    embedded_df = standardized_df.copy()
    embedded_df["skill_embedding"] = [embedding.tolist() for embedding in embeddings]
    embedded_df.to_parquet(output_file, engine="pyarrow")

    print(f"Saved {len(embedded_df):,} rows to {output_file}")
    return embedded_df


In [ ]:
print("Loading MPNet embedding model...")
embedding_model = SentenceTransformer(resolve_mpnet_model_path())


## NUS Modules

In [ ]:
nus_embedded = build_embedded_modules(
    csv_path=PROJECT_DIR / "all_nus_modules.csv",
    output_file=PROJECT_DIR / "all_nus_modules_embedded.parquet",
    university="nus",
    embedding_model=embedding_model,
)

nus_embedded.head()


## SMU Modules

In [ ]:
smu_embedded = build_embedded_modules(
    csv_path=PROJECT_DIR / "all_smu_modules.csv",
    output_file=PROJECT_DIR / "all_smu_modules_embedded.parquet",
    university="smu",
    embedding_model=embedding_model,
)

smu_embedded.head()


## SUTD Modules

In [ ]:
sutd_embedded = build_embedded_modules(
    csv_path=PROJECT_DIR / "all_sutd_modules.csv",
    output_file=PROJECT_DIR / "all_sutd_modules_embedded.parquet",
    university="sutd",
    embedding_model=embedding_model,
)

sutd_embedded.head()


## All Schools Combined

In [ ]:
all_schools_embedded = pd.concat(
    [nus_embedded, smu_embedded, sutd_embedded],
    ignore_index=True,
)

all_schools_output = PROJECT_DIR / "all_schools_modules_embedded.parquet"
all_schools_embedded.to_parquet(all_schools_output, engine="pyarrow")

print(f"Saved {len(all_schools_embedded):,} rows to {all_schools_output}")
all_schools_embedded.head()
